In [2]:
from notebookutils import mssparkutils
from datetime import datetime

try:
    batch_id = mssparkutils.widgets.get("batch_id")
except Exception:
    batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
print("batch_id = ", batch_id)

StatementMeta(, 52c055b1-d347-47d4-84cb-2502f1fb733b, 4, Finished, Available, Finished, True)

batch_id =  20260416_181417


In [2]:
from pyspark.sql import functions as F

raw_tip_path = "Files/yelp/raw/yelp_tip.json"
df_tip_raw = spark.read.json(raw_tip_path)
df_tip_raw.printSchema()
print("raw tip count:", df_tip_raw.count())


StatementMeta(, 132cc1ec-bb0e-4f22-9506-683dd6885db4, 4, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- compliment_count: long (nullable = true)
 |-- date: string (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)

raw tip count: 908915


In [5]:
df_tip_bronze = (
    df_tip_raw
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file",F.input_file_name())
    .withColumn("_batch_id",F.lit(batch_id))
)

StatementMeta(, 132cc1ec-bb0e-4f22-9506-683dd6885db4, 7, Finished, Available, Finished, False)

In [6]:
(
    df_tip_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("tip_bronze")
)

StatementMeta(, 132cc1ec-bb0e-4f22-9506-683dd6885db4, 8, Finished, Available, Finished, False)

In [7]:
df_tip_bronze.select("business_id","compliment_count","date","text","user_id")


StatementMeta(, 132cc1ec-bb0e-4f22-9506-683dd6885db4, 9, Finished, Available, Finished, False)

DataFrame[business_id: string, compliment_count: bigint, date: string, text: string, user_id: string]

In [8]:
spark.table("tip_bronze").printSchema()

StatementMeta(, 132cc1ec-bb0e-4f22-9506-683dd6885db4, 10, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- compliment_count: long (nullable = true)
 |-- date: string (nullable = true)
 |-- text: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [9]:
#Return to 01_0_bronze_run_all
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, 132cc1ec-bb0e-4f22-9506-683dd6885db4, 11, Finished, Available, Finished, False)

ExitValue: SUCCESS